# Deliverable 4 — Evaluación Oficial Multi-LoRA (`3-LoRA` vs `Control Base`, `75` + `Nuevas 20`)
Ejecuta y verifica en GPU Colab la arquitectura oficial de **Deliverable 4** cargando los adaptadores públicos desde Hugging Face (`--solo-hf`):
- **Sistema Oficial (`--agentes enrutador,recuperador,redactor`)**: los 3 adaptadores QLoRA cargados sobre el mismo `Qwen/Qwen3-4B` NF4 con conmutación dinámica vía `set_adapter()`, manteniendo al `Verificador` con pesos Base (`disable_adapter()`).
- **Control Base pareado (`--agentes ninguno`)**: exactamente el mismo binario, cuantización, índice E5 y semillas con los 4 agentes en pesos Base (`disable_adapter()`).
- **Reproducción rápida desde HF:** por defecto valida el manifiesto `manifest_adaptadores_hf.json` y corre con `--solo-hf` sin depender de Google Drive para los pesos.

In [ ]:
import subprocess
subprocess.run(["nvidia-smi", "-L"], check=False)
%pip -q install -U transformers==5.23.0 peft==0.21.3 bitsandbytes==0.50.2 accelerate==1.15.0 sentence-transformers==6.3.0 pandas matplotlib huggingface-hub==1.33.0

import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    bf16_ok = torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
    print(f"GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | bf16={bf16_ok}")

## Paso 2 — Clonar rama `deliverable-4` y verificar adaptadores en Hugging Face

In [ ]:
from pathlib import Path
import subprocess

MONTAR_DRIVE_PARA_RESULTADOS = True
SOLO_HF = True  # True: descarga siempre adaptadores desde Hugging Face verificando manifest

if MONTAR_DRIVE_PARA_RESULTADOS:
    from google.colab import drive
    drive.mount("/content/drive")

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
RAMA = "deliverable-4"
WORKDIR = Path("/content/Proyecto_GenIA")

if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", RAMA], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "checkout", RAMA], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", RAMA], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, REPO_URL, str(WORKDIR)], check=True)

D4_DIR = WORKDIR / "Deliverable 4"
SCRIPT_EVAL = D4_DIR / "scripts" / "evaluar_multi_lora.py"
SCRIPT_RESUMEN = D4_DIR / "scripts" / "resumir_comparativa.py"
SCRIPT_PUBLICAR = D4_DIR / "scripts" / "publicar_adaptadores_hf.py"
NUEVAS_20_CSV = D4_DIR / "datos" / "preguntas_nuevas_20.csv"
for p in (SCRIPT_EVAL, SCRIPT_RESUMEN, SCRIPT_PUBLICAR, NUEVAS_20_CSV):
    assert p.is_file(), f"Falta archivo requerido: {p}"

FLAGS_HF = ["--solo-hf"] if SOLO_HF else []
print("Commit clonado:", subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "--short", "HEAD"], text=True).strip())
subprocess.run(["python3", str(SCRIPT_PUBLICAR), "--verificar-manifest"], check=True)

## Paso 3 — Smoke test de conmutación y aislamiento (`congelado_50:13`)

In [ ]:
from datetime import datetime, timezone
import json
import subprocess
from pathlib import Path

REANUDAR_CARPETA = None  # Pega ruta si deseas reanudar una corrida previa
if REANUDAR_CARPETA:
    OUT_DIR = Path(REANUDAR_CARPETA)
else:
    STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    BASE_OUT = Path("/content/drive/MyDrive/D4_eval_3lora") if MONTAR_DRIVE_PARA_RESULTADOS else Path("/content/D4_eval_3lora")
    OUT_DIR = BASE_OUT / f"run_{STAMP}"

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Carpeta de salida:", OUT_DIR)

SMOKE_DIR = OUT_DIR / "_smoke"
cmd_smoke = [
    "python3", str(SCRIPT_EVAL),
    "--agentes", "enrutador,recuperador,redactor",
    *FLAGS_HF,
    "--output-dir", str(SMOKE_DIR),
    "--output-prefix", "smoke_3lora",
    "--qids", "congelado_50:13",
    "--verbose-pasos",
]
subprocess.run(cmd_smoke, check=True)

tel_smoke = json.loads((SMOKE_DIR / "telemetria_smoke_3lora.json").read_text(encoding="utf-8"))["congelado_50:13"]
for c in tel_smoke:
    print(" ", f"{c['agente']:<11}", "| usa_adaptador=", c["usa_adaptador"], "| adaptador=", c.get("adaptador_activo"), "| s=", c["segundos"])

## Paso 4 — Reproducción rápida desde HF (`7` preguntas de la comparativa pareada)
Permite confirmar en pocos minutos que los adaptadores publicados en Hugging Face reproducen la comparativa Base vs `3-LoRA`.

In [ ]:
import subprocess

QIDS_COMP = "dificil_25:H07,dificil_25:H14,congelado_50:11,congelado_50:43,dificil_25:H18,dificil_25:H24,dificil_25:H03"
cmd_comp = [
    "python3", str(SCRIPT_EVAL),
    "--agentes", "enrutador,recuperador,redactor",
    *FLAGS_HF,
    "--output-dir", str(OUT_DIR),
    "--solo-comparativa",
    "--comparar-vs-base", QIDS_COMP,
    "--verbose-pasos",
]
subprocess.run(cmd_comp, check=True)

## Paso 5 — Evaluar las `75` preguntas oficiales con **D4 `3-LoRA` Oficial**

In [ ]:
import subprocess

cmd_3lora_75 = [
    "python3", str(SCRIPT_EVAL),
    "--agentes", "enrutador,recuperador,redactor",
    *FLAGS_HF,
    "--output-dir", str(OUT_DIR),
    "--output-prefix", "3lora_75",
]
print("Ejecutando 3-LoRA (75):", " ".join(cmd_3lora_75))
subprocess.run(cmd_3lora_75, check=True)

## Paso 6 — Evaluar **Control Base pareado** (`--agentes ninguno`) sobre las `75` preguntas

In [ ]:
import subprocess

cmd_base_75 = [
    "python3", str(SCRIPT_EVAL),
    "--agentes", "ninguno",
    *FLAGS_HF,
    "--output-dir", str(OUT_DIR),
    "--output-prefix", "control_base_75",
]
print("Ejecutando Control Base (75):", " ".join(cmd_base_75))
subprocess.run(cmd_base_75, check=True)

## Paso 7 — Evaluar `preguntas_nuevas_20.csv` y generar resumen automático

In [ ]:
import subprocess

for agentes, pref in (("enrutador,recuperador,redactor", "3lora_nuevas20"), ("ninguno", "control_base_nuevas20")):
    cmd = [
        "python3", str(SCRIPT_EVAL),
        "--agentes", agentes,
        *FLAGS_HF,
        "--preguntas-csv", str(NUEVAS_20_CSV),
        "--nombre-set", "nuevas_20",
        "--output-dir", str(OUT_DIR),
        "--output-prefix", pref,
    ]
    print(f"Ejecutando Nuevas 20 ({pref}):", " ".join(cmd))
    subprocess.run(cmd, check=True)

for bench, csv_name, tel_name, out_name in (
    ("75", "resultados_3lora_75.csv", "telemetria_3lora_75.json", "resumen_comparativo_75.json"),
    ("nuevas_20", "resultados_3lora_nuevas20.csv", "telemetria_3lora_nuevas20.json", "resumen_comparativo_nuevas20.json"),
):
    subprocess.run(
        [
            "python3", str(SCRIPT_RESUMEN),
            "--benchmark", bench,
            "--multi-csv", str(OUT_DIR / csv_name),
            "--multi-telemetria", str(OUT_DIR / tel_name),
            "--salida-json", str(OUT_DIR / out_name),
        ],
        check=True,
    )

## Paso 8 — Empaquetar y descargar resultados completos de evaluación `3-LoRA`

In [ ]:
import zipfile
from google.colab import files

bundle_zip = OUT_DIR / "D4_resultados_3lora.zip"
patrones = ("resultados_*.csv", "trazas_*.json", "telemetria_*.json", "configuracion_*.json", "comparativa_*.json", "resumen_*.json")
archivos = sorted({p for pat in patrones for p in OUT_DIR.glob(pat) if p.is_file()})
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in archivos:
        zf.write(f, arcname=f.name)

print(f"Paquete generado ({len(archivos)} archivos):", bundle_zip)
files.download(str(bundle_zip))